# 01. Data Collection — 공공데이터 수집 및 정제
**산출물**: core_articles.json, violation_mapping.json, severity_6levels.json, aggravating_factors.json

**의존**: 00_setup.py 완료, law_collector.py 존재

In [13]:
import os, sys, pathlib, json, glob
import pandas as pd
from dotenv import load_dotenv
load_dotenv()

from config import (load_korean_csv, save_json, DTG_DIR, SPEED_DIR,
                    LAW_DIR, JSON_DIR, PENALTY_DIR,
                    SEVERITY_TABLE, AGGRAVATING_FACTORS, VIOLATION_MAPPING)

print('📁 데이터 폴더 확인')
for n, p in [('DTG',DTG_DIR),('제한속도',SPEED_DIR),('법령',LAW_DIR)]:
    cnt = len(list(p.glob('*'))) if p.exists() else 0
    print(f'  {"✅" if cnt else "❌"} {n}: {p} ({cnt}개)')

📁 데이터 폴더 확인
  ✅ DTG: data\01_DTG_RAW (5개)
  ✅ 제한속도: data\02_SPEED_LIMIT (2개)
  ✅ 법령: data\03_LAW_DATA (15개)


## 1. DTG 데이터 로딩

In [14]:
dtg_files = sorted(glob.glob(str(DTG_DIR/'*.csv')))
print(f'=== DTG 파일 ({len(dtg_files)}개) ===')

codebook_file = daily_file = trip_file = body_file = header_file = None
for f in dtg_files:
    nm = pathlib.Path(f).name; nl = nm.lower()
    if 'codebook' in nl or '코드' in nm: codebook_file = f
    elif 'dt_car' in nl or 'bsn_ddra' in nl: daily_file = f
    elif 'trip' in nl or 'Trip' in nm: trip_file = f
    elif '바디' in nm or 'body' in nl: body_file = f
    elif '헤더' in nm or 'header' in nl: header_file = f

for lb, fp in [('Codebook',codebook_file),('Daily집계',daily_file),
               ('Trip단위',trip_file),('초단위바디',body_file),('초단위헤더',header_file)]:
    print(f'  {lb:12s}: {"✅ "+pathlib.Path(fp).name if fp else "❌"}')

=== DTG 파일 (5개) ===
  Codebook    : ✅ CODEBOOK.csv
  Daily집계     : ✅ DT_CAR_BSN_DDRA_202507311033.csv
  Trip단위      : ✅ 사업용차량 Trip단위 위험운전운행데이터_샘플.csv
  초단위바디       : ✅ 사업용차량 초단위 운행기록데이터_샘플(body).csv
  초단위헤더       : ✅ 사업용차량 초단위 운행기록데이터_샘플(header).csv


In [15]:
if daily_file:
    df_daily = load_korean_csv(daily_file)
    print(f'일별 집계: {df_daily.shape}')
    print(df_daily.head(2).to_string())

if trip_file:
    df_trip = load_korean_csv(trip_file)
    print(f'\nTrip단위: {df_trip.shape}, 컬럼 {len(df_trip.columns)}개')

if body_file:
    df_body = load_korean_csv(body_file)
    print(f'\n초단위 바디: {df_body.shape}')
    print(df_body.head(3).to_string())

일별 집계: (28654, 19)
         평가일자  업종코드  분기코드  계절코드  요일코드  과속건수20키로미터초과  장기과속건수    급가속건수  급출발건수   급감속건수   급정지건수  급좌회전건수  급우회전건수  급유턴건수  급앞지르기건수  급진로변경건수  위험운전행동건수     운행거리        생성일시
0  2020-07-01    11     3     2     4        404850      26  1405623   4062  991162  249722  373125  237844  98736   100488   491962   4357600  7713247  2023-08-01
1  2020-07-01    12     3     2     4         65490       0    25764      5   36287   14939   28361   18356   7307     2286    11520    210315   519457  2023-08-01

Trip단위: (100, 66), 컬럼 66개

초단위 바디: (1100, 14)
  운행기록데이터 바디 항목  일일주행거리(KM)  누적운행거리(KM)            정보발생일시  운행속도(KMH)   RPM  브레이크신호  시작GPS(X좌표)  시작GPS(Y좌표)  GPS방위각  가속도 Vx  가속도 Vy  통신상태코드      운행지역코드
0    운행기록데이터 바디           7       97703  2025020608272200         43  1059       1   126889400    37652552     257    -1.2    -1.8       0  4128111100
1    운행기록데이터 바디           7       97703  2025020608272300         41  1000       1   126889272    37652532     256    -1.2    -2.5       0  

## 2. 제한속도 데이터

In [16]:
speed_files = sorted(glob.glob(str(SPEED_DIR/'*.csv')))
print(f'=== 제한속도 ({len(speed_files)}개) ===')
for sf in speed_files:
    df_sp = load_korean_csv(sf, nrows=5000)
    fn = pathlib.Path(sf).name
    sc = [c for c in df_sp.columns if '속도' in c or '제한' in c]
    print(f'\n{fn}: {df_sp.shape}')
    if sc:
        print(f'  속도컬럼: {sc}')
        print(f'  분포:\n{df_sp[sc[0]].value_counts().head(5).to_string()}')

=== 제한속도 (2개) ===

전국도로안전표지표준데이터.csv: (5000, 24)
  속도컬럼: ['주행제한속도']
  분포:
주행제한속도
30.0    238
60.0    218
50.0    213
70.0    170
80.0     55

한국도로공사_고속도로 구간별 제한속도_20250501.csv: (95, 6)
  속도컬럼: ['기점 방향 제한속도(kph)', '종점 방향 제한속도(kph)']
  분포:
기점 방향 제한속도(kph)
100    69
80     14
110    11
120     1


## 3. 법령 원문 수집 (법제처 Open API)
수집 대상: 도로교통법 10조 + 시행규칙 1조 + 교통안전법 2조 + 화물차법 2조 = 15개
(교통안전법 제56조는 2024년 개정으로 교통안전체험시설 조문이 되어 제외)

In [17]:
from law_collector import LawCollector, ARTICLE_META, ensure_dirs

ensure_dirs()
print(f'수집 대상: {len(ARTICLE_META)}개 조문')
for m in ARTICLE_META:
    print(f'  {m["id"]:15s} | {m["law"]:20s} | {m["jo_num"]:12s} | {m["title"]}')

수집 대상: 15개 조문
  RTA_17          | 도로교통법                | 제17조         | 자동차등의 속도
  RTA_12          | 도로교통법                | 제12조         | 어린이 보호구역의 지정 및 관리
  RTA_19          | 도로교통법                | 제19조         | 안전거리 확보 등
  RTA_21          | 도로교통법                | 제21조         | 앞지르기 방법 등
  RTA_25          | 도로교통법                | 제25조         | 교차로 통행방법
  RTA_49          | 도로교통법                | 제49조         | 모든 차의 운전자의 준수사항 등
  RTA_151_2       | 도로교통법                | 제151조의2      | 벌칙 (난폭운전·반복초과속)
  RTA_153         | 도로교통법                | 제153조        | 벌칙 (100만원 이하)
  RTA_154         | 도로교통법                | 제154조        | 벌칙 (30만원 이하)
  RTA_156         | 도로교통법                | 제156조        | 벌칙 (20만원 이하)
  RTA_R_19        | 도로교통법 시행규칙           | 제19조         | 자동차등과 노면전차의 속도
  TSA_54_2        | 교통안전법                | 제54조의2       | 교통안전담당자 지정
  TSA_55          | 교통안전법                | 제55조         | 운행기록장치의 장착 등
  TTBA_11         | 화물자동차 운수사업법          | 제11조         | 허가취소 

In [18]:
import urllib.request

ip = urllib.request.urlopen("https://api.ipify.org").read().decode()
print(ip)

121.141.17.74


In [19]:
collector = LawCollector()
CORE_ARTICLES = collector.collect_all()
collector.validate()
collector.save()
collector.save_by_law()

print(f'\n=== 수집 결과 ===')
for a in CORE_ARTICLES:
    c = a.get('content','')
    preview = c[:50]+'...' if len(c) > 50 else c
    print(f'  {"✅" if c else "❌"} {a["id"]:15s}: {preview}')

2026-04-12 23:36:01,219 [INFO] 
2026-04-12 23:36:01,220 [INFO] [Step 1] 법령 MST 조회 (4개)
2026-04-12 23:36:01,220 [INFO] ============================================================
2026-04-12 23:36:01,651 [INFO]   ✅ 도로교통법: MST=281875
2026-04-12 23:36:02,097 [INFO]   ✅ 도로교통법 시행규칙: MST=285317
2026-04-12 23:36:02,529 [INFO]   ✅ 교통안전법: MST=254099
2026-04-12 23:36:02,952 [INFO]   ✅ 화물자동차 운수사업법: MST=273303
2026-04-12 23:36:02,953 [INFO] 
2026-04-12 23:36:02,953 [INFO] [Step 2] 조문 수집 (15개)
2026-04-12 23:36:02,954 [INFO] ============================================================
2026-04-12 23:36:04,050 [INFO]   📥 도로교통법: 전체 229개 조문 로드
2026-04-12 23:36:05,565 [INFO]   📥 도로교통법 시행규칙: 전체 212개 조문 로드
2026-04-12 23:36:06,518 [INFO]   📥 교통안전법: 전체 82개 조문 로드
2026-04-12 23:36:07,554 [INFO]   📥 화물자동차 운수사업법: 전체 137개 조문 로드
2026-04-12 23:36:07,554 [INFO]   ✅ RTA_17          제17조        : ① 자동차등(개인형 이동장치는 제외한다. 이하 이 조에서 같다)과 노면전차의 도로 통행 속도는 행정... (정확매칭)
2026-04-12 23:36:09,926 [INFO]   🔄 RTA_12          제12조  


=== 수집 결과 ===
  ✅ RTA_17         : ① 자동차등(개인형 이동장치는 제외한다. 이하 이 조에서 같다)과 노면전차의 도로 통행 속...
  ✅ RTA_12         : ① 시장등은 교통사고의 위험으로부터 어린이를 보호하기 위하여 필요하다고 인정하는 경우에는 ...
  ✅ RTA_19         : ① 모든 차의 운전자는 같은 방향으로 가고 있는 앞차의 뒤를 따르는 경우에는 앞차가 갑자기...
  ✅ RTA_21         : ① 모든 차의 운전자는 다른 차를 앞지르려면 앞차의 좌측으로 통행하여야 한다. ② 자전거등...
  ✅ RTA_25         : ① 모든 차의 운전자는 교차로에서 우회전을 하려는 경우에는 미리 도로의 우측 가장자리를 서...
  ✅ RTA_49         : ① 모든 차 또는 노면전차의 운전자는 다음 각 호의 사항을 지켜야 한다. <개정 2013....
  ✅ RTA_151_2      : 제151조의2(벌칙) 다음 각 호의 어느 하나에 해당하는 사람은 1년 이하의 징역이나 50...
  ✅ RTA_153        : ①다음 각 호의 어느 하나에 해당하는 사람은 6개월 이하의 징역이나 200만원 이하의 벌금...
  ✅ RTA_154        : 제154조(벌칙) 다음 각 호의 어느 하나에 해당하는 사람은 30만원 이하의 벌금이나 구류...
  ✅ RTA_156        : 제156조(벌칙) 다음 각 호의 어느 하나에 해당하는 사람은 20만원 이하의 벌금이나 구류...
  ✅ RTA_R_19       : 자동차등의 도로 통행 최고속도는 다음과 같다: 고속도로 승용자동차 100~110km/h, ...
  ✅ TSA_54_2       : 대통령령으로 정하는 교통수단 운영자는 교통안전에 관한 업무를 담당할 교통안전담당자를 해당 ...
  ✅ TSA_55         : ① 교통수단 운영자는 대통령령으로 정하는 교통수단에 운행기록장치(DTG)를 장착하여야 한다...
  ✅ TTBA_1

In [20]:
# ★★ 조문 내용 강제 보정 패치
# law_collector.py 버전과 무관하게 노트북 자체에서 4개 조문을 보장합니다.
# - RTA_151_2: API가 80자 짧은 내용 반환 → 전문으로 교체
# - TTBA_11:   API가 개정이력 문자열(75자) 반환 → 실제 조문으로 교체
# - TTBA_59:   API가 개정이력 문자열(75자) 반환 → 실제 조문으로 교체

import re as _re
_REVISION_RE = _re.compile(r'^\d{8}:')

ARTICLE_OVERRIDE = {
    'RTA_151_2': (
        '제151조의2(벌칙) 다음 각 호의 어느 하나에 해당하는 사람은 1년 이하의 징역이나 '
        '500만원 이하의 벌금에 처한다. '
        '제1호: 제46조의3을 위반하여 난폭운전을 한 사람. '
        '제2호: 제17조제3항을 위반하여 자동차등의 최고속도보다 시속 100킬로미터를 초과한 '
        '속도로 3회 이상 운전한 사람. '
        '반복 초과속은 교통안전을 중대하게 위협하므로 형사처벌 대상이다. <개정 2020.6.9>'
    ),
    'TTBA_11': (
        '① 국토교통부장관은 화물자동차 운송사업자가 다음 각 호의 어느 하나에 해당하면 '
        '허가를 취소하거나 6개월 이내의 기간을 정하여 사업의 전부 또는 일부의 정지를 명할 수 있다. '
        '제5호: 운수종사자가 이 법 또는 도로교통법을 위반하여 운전면허가 취소된 경우. '
        '제12호: 이 법 또는 이 법에 따른 명령이나 처분을 위반한 경우. '
        '② 제1항에도 불구하고 제1항제1호에 해당하는 경우에는 허가를 취소하여야 한다. '
        '과속·난폭운전 등 중대한 교통법규 위반이 반복되는 운수종사자를 고용한 사업자는 '
        '사업허가 취소 또는 정지 처분의 대상이 된다. (화물자동차 운수사업법 제11조)'
    ),
    'TTBA_59': (
        '① 화물자동차 운수사업자는 소속 운수종사자에게 교통안전에 관한 교육을 실시하여야 한다. '
        '② 운수종사자는 국토교통부령으로 정하는 바에 따라 '
        '교통안전체험에 관한 교육을 매년 4시간 이상 이수하여야 한다. '
        '③ 제2항에 따른 교육을 받지 아니한 운수종사자를 운행에 종사하게 한 자 및 '
        '교육을 받지 아니한 운수종사자에게는 100만원 이하의 과태료를 부과한다. '
        '위험운전행동이 반복적으로 기록된 운수종사자는 교육 이수 대상에 우선 포함된다. '
        '(화물자동차 운수사업법 제59조)'
    ),
}

# ★★ 벌칙 조문 호(號) 보강: API는 조문 본문만 반환하고 개별 호를 미포함.
# KG에서 '80km/h 초과 → 제154조제9호' 연결을 만들려면 호 텍스트가 필수.
# 출처: 국가법령정보센터(law.go.kr) 도로교통법(법률 제21016호, 2026.01.01 시행)
PENALTY_ARTICLE_SUPPLEMENT = {
    'RTA_153': (
        ' 제2항제2호: 제17조제3항을 위반하여 자동차등의 최고속도보다 '
        '시속 100킬로미터를 초과한 속도로 운전한 사람.'
    ),
    'RTA_154': (
        ' 제9호: 제17조제3항을 위반하여 최고속도보다 '
        '시속 80킬로미터를 초과한 속도로 운전한 사람.'
    ),
    'RTA_156': (
        ' 제1호: 제17조제3항을 위반하여 제한속도를 초과하여 운전한 사람. '
        '과속 범칙금은 시행령 별표 8에 따라 초과속도 구간별 차등 적용.'
    ),
}

patched = []
# (A) 100자 미만 또는 개정이력 → 전체 교체
for art in CORE_ARTICLES:
    art_id = art['id']
    current = art.get('content', '').strip()
    need_patch = len(current) < 100 or _REVISION_RE.match(current)
    if need_patch and art_id in ARTICLE_OVERRIDE:
        art['content'] = ARTICLE_OVERRIDE[art_id]
        art['collect_method'] = 'notebook_patch'
        patched.append(art_id)

# (B) 벌칙 조문 호 보강: API 원문 + 호 텍스트 추가
supplemented = []
for art in CORE_ARTICLES:
    art_id = art['id']
    if art_id in PENALTY_ARTICLE_SUPPLEMENT:
        supp = PENALTY_ARTICLE_SUPPLEMENT[art_id]
        # 이미 호 텍스트가 포함되어 있으면 스킵
        if supp.strip()[:10] not in art.get('content',''):
            art['content'] = art['content'].rstrip() + supp
            art['collect_method'] = 'exact_match+supplement'
            supplemented.append(art_id)

if patched:
    print(f'✅ 조문 내용 패치 완료: {patched}')
    for art in CORE_ARTICLES:
        if art['id'] in patched:
            print(f'   {art["id"]}: {len(art["content"])}자 → collect_method=notebook_patch')
if supplemented:
    print(f'✅ 벌칙 조문 호 보강 완료: {supplemented}')
    for art in CORE_ARTICLES:
        if art['id'] in supplemented:
            print(f'   {art["id"]}: {len(art["content"])}자 → 호 텍스트 추가')
if patched or supplemented:
    # ★★ 패치/보강 결과를 JSON에 재저장 (03_kg_construction이 디스크 파일을 읽으므로 필수)
    save_json(CORE_ARTICLES, JSON_DIR / 'core_articles.json')
    # 법령별 JSON도 재저장
    from collections import defaultdict as _dd
    _by_law = _dd(list)
    for a in CORE_ARTICLES: _by_law[a['law']].append(a)
    for _ln, _arts in _by_law.items():
        _fp = JSON_DIR / f'{_ln.replace(" ","_")}.json'
        save_json(_arts, _fp)
else:
    print('ℹ️  모든 조문 정상 수집됨 (패치/보강 불필요)')


✅ 조문 내용 패치 완료: ['RTA_151_2']
   RTA_151_2: 212자 → collect_method=notebook_patch
✅ 벌칙 조문 호 보강 완료: ['RTA_153', 'RTA_154', 'RTA_156']
   RTA_153: 216자 → 호 텍스트 추가
   RTA_154: 191자 → 호 텍스트 추가
   RTA_156: 336자 → 호 텍스트 추가
✅ 저장: data\03_LAW_DATA\json\core_articles.json
✅ 저장: data\03_LAW_DATA\json\도로교통법.json
✅ 저장: data\03_LAW_DATA\json\도로교통법_시행규칙.json
✅ 저장: data\03_LAW_DATA\json\교통안전법.json
✅ 저장: data\03_LAW_DATA\json\화물자동차_운수사업법.json


## 4. 법령 PDF 확인

In [21]:
pdf_files = sorted(LAW_DIR.glob('*.pdf'))
print(f'=== 법령 PDF ({len(pdf_files)}개) ===')
for pf in pdf_files:
    print(f'  {pf.name} ({pf.stat().st_size/1024:.0f}KB)')

=== 법령 PDF (13개) ===
  [별표 10] 어린이보호구역 및 노인ㆍ장애인보호구역에서의 범칙행위 및 범칙금액(제93조제2항 관련)(도로교통법 시행령).pdf (54KB)
  [별표 28] 운전면허 취소·정지처분 기준(제91조제1항관련)(도로교통법 시행규칙).pdf (141KB)
  [별표 7] 어린이보호구역 및 노인ㆍ장애인보호구역에서의 과태료 부과기준(제88조제4항 단서 관련)(도로교통법 시행령).pdf (49KB)
  [별표 8] 범칙행위 및 범칙금액(운전자)(제93조제1항 관련)(도로교통법 시행령).pdf (93KB)
  교통안전법 시행규칙(국토교통부령)(제01411호)(20241129).pdf (151KB)
  교통안전법 시행령(대통령령)(제35947호)(20260102).pdf (161KB)
  교통안전법(법률)(제19673호)(20240817).pdf (165KB)
  도로교통법 시행규칙(행정안전부령)(제00610호)(20260224).pdf (322KB)
  도로교통법 시행령(대통령령)(제35947호)(20260102).pdf (266KB)
  도로교통법(법률)(제21016호)(20260101).pdf (382KB)
  화물자동차 운수사업법 시행규칙(국토교통부령)(제01551호)(20251229).pdf (237KB)
  화물자동차 운수사업법 시행령(대통령령)(제35832호)(20260101).pdf (172KB)
  화물자동차 운수사업법(법률)(제21025호)(20260101).pdf (264KB)


## 5. SGT 저장 (과속 6단계 + 가중요인 + 위반매핑)

In [22]:
save_json(SEVERITY_TABLE, PENALTY_DIR / 'severity_6levels.json')
save_json(AGGRAVATING_FACTORS, PENALTY_DIR / 'aggravating_factors.json')
save_json(VIOLATION_MAPPING, JSON_DIR / 'violation_article_mapping.json')

print('\n=== 과속 법적 심각도 6단계 ===')
for lv, info in SEVERITY_TABLE.items():
    tag = '🔴형사' if info['criminal'] else '🟢행정'
    dp = info.get('demerit_points')
    print(f"  {lv}: {info['description']:20s} | {tag} | 벌점 {str(dp)+'점' if dp is not None else '취소':5s} | {info['legal_basis']}")

print(f'\n위반유형 {len(VIOLATION_MAPPING)}개 매핑 완료')

✅ 저장: data\03_LAW_DATA\penalty_tables\severity_6levels.json
✅ 저장: data\03_LAW_DATA\penalty_tables\aggravating_factors.json
✅ 저장: data\03_LAW_DATA\json\violation_article_mapping.json

=== 과속 법적 심각도 6단계 ===
  level_1: 20km/h 이하 초과         | 🟢행정 | 벌점 0점    | 제156조제1호
  level_2: 20~40km/h 초과         | 🟢행정 | 벌점 15점   | 제156조제1호
  level_3: 40~60km/h 초과         | 🟢행정 | 벌점 30점   | 제156조제1호
  level_4: 60~80km/h 초과         | 🟢행정 | 벌점 60점   | 제156조제1호
  level_5: 80~100km/h 초과        | 🔴형사 | 벌점 80점   | 제154조제9호
  level_6: 100km/h 초과           | 🔴형사 | 벌점 취소    | 제153조제2항제2호

위반유형 10개 매핑 완료


In [23]:
# ★★ VIOLATION_MAPPING 보완 — dtg_violation, education, overload 추가
# config.py의 기본 VIOLATION_MAPPING에 없는 교차 법령 위반유형을 JSON에 직접 추가
# (G 카테고리 시나리오의 M6 Multi-hop Recall 계산에 필수)

import json as _json
from pathlib import Path as _Path
from config import JSON_DIR

_vm_path = _Path(JSON_DIR) / 'violation_article_mapping.json'
with open(_vm_path, 'r', encoding='utf-8') as _f:
    _vmap = _json.load(_f)

_extra = {
    'dtg_violation': {
        'name_kr': 'DTG 운행기록 미제출',
        'etas_threshold': '운행기록 미제출 또는 불성실 제출',
        'definition_articles': ['TSA_55'],  # TSA_56은 법령 개정으로 교통안전체험시설 조문이 됨. 제출의무는 TSA_55 제2항에 포함.
        'penalty_articles': [],
        'has_severity_levels': False,
        'note': '교통안전법 제55조 위반 — 과태료. 심각도 등급 없음'
    },
    'education': {
        'name_kr': '운수종사자 교육 미이수',
        'etas_threshold': '연간 4시간 교통안전체험교육 미이수',
        'definition_articles': ['TTBA_59'],
        'penalty_articles': [],
        'has_severity_levels': False,
        'note': '화물자동차 운수사업법 제59조 위반 — 100만원 이하 과태료'
    },
    'overload': {
        'name_kr': '적재량 초과',
        'etas_threshold': '최대 적재량 초과 운행',
        'definition_articles': ['RTA_49'],
        'penalty_articles': ['RTA_156'],
        'has_severity_levels': False,
        'note': '도로교통법 제49조 위반 — 범칙금 처분'
    },
}

_added = []
for _k, _v in _extra.items():
    if _k not in _vmap:
        _vmap[_k] = _v
        _added.append(_k)

with open(_vm_path, 'w', encoding='utf-8') as _f:
    _json.dump(_vmap, _f, ensure_ascii=False, indent=2)

print(f'✅ violation_article_mapping: {len(_vmap)}개 위반유형')
if _added:
    print(f'   추가됨: {_added}')
else:
    print('   (이미 추가되어 있음)')


✅ violation_article_mapping: 13개 위반유형
   추가됨: ['dtg_violation', 'education', 'overload']


## 6. 완료 요약

In [24]:
print('='*60)
print('  01_data_collection 완료')
print('='*60)
print(f'  조문: {len(CORE_ARTICLES)}개')
print(f'  SGT: {len(SEVERITY_TABLE)}단계')

import json as _j
from config import JSON_DIR
_vmap_final = _j.load(open(JSON_DIR / 'violation_article_mapping.json', encoding='utf-8'))
print(f'  위반매핑: {len(_vmap_final)}개 (도로교통법+교통안전법+화물차법 포함)')

# ★ 품질 검증: 100자 이상 + 개정이력 패턴 거부
import re as _re
_REV = _re.compile(r'^\d{8}:')
print(f'\n=== 조문 수집 품질 ===')
bad = []
for art in CORE_ARTICLES:
    c = art.get('content', '').strip()
    is_rev = bool(_REV.match(c))
    ok = len(c) >= 100 and not is_rev
    tag = '✅' if ok else ('⚠️개정이력' if is_rev else '❌')
    if not ok: bad.append(art['id'])
    print(f"  {tag} {art['id']:15s} {art['jo_num']:12s}: {len(c):4d}자")

if bad:
    print(f'\n❌ 미완료 조문: {bad}')
    raise RuntimeError(f'조문 수집 실패 — 03_kg_construction 실행 불가. law_collector.py 확인 필요.')
else:
    print(f'\n✅ 전체 {len(CORE_ARTICLES)}개 조문 100자 이상 수집 완료')
    print(f'\n→ 다음: 02_data_analysis.ipynb')


  01_data_collection 완료
  조문: 15개
  SGT: 6단계
  위반매핑: 13개 (도로교통법+교통안전법+화물차법 포함)

=== 조문 수집 품질 ===
  ✅ RTA_17          제17조        :  427자
  ✅ RTA_12          제12조        :  146자
  ✅ RTA_19          제19조        :  410자
  ✅ RTA_21          제21조        :  468자
  ✅ RTA_25          제25조        :  858자
  ✅ RTA_49          제49조        :  297자
  ✅ RTA_151_2       제151조의2     :  212자
  ✅ RTA_153         제153조       :  216자
  ✅ RTA_154         제154조       :  191자
  ✅ RTA_156         제156조       :  336자
  ✅ RTA_R_19        제19조        :  172자
  ✅ TSA_54_2        제54조의2      :  123자
  ✅ TSA_55          제55조        :  181자
  ✅ TTBA_11         제11조        : 2687자
  ✅ TTBA_59         제59조        :  426자

✅ 전체 15개 조문 100자 이상 수집 완료

→ 다음: 02_data_analysis.ipynb
